# Semantic Patent Retrieval: BM25 vs Base vs Fine-tuned Transformers (Colab)
**Student:** X24244732 | **Programme:** MSCAIJAN26I

Compares five retrieval systems on the DAPFAM benchmark (IPC sections G and H), loaded online from the Hugging Face Hub:

| System | Type |
|---|---|
| BM25 (k1 = 1.2, b = 0.75, Porter stemming) | lexical baseline |
| all-MiniLM-L6-v2 (base) | general-purpose sentence transformer |
| all-MiniLM-L6-v2 (fine-tuned) | general model, domain-adapted |
| PatentSBERTa (base) | patent-specific sentence transformer |
| PatentSBERTa (fine-tuned) | patent model, domain-adapted |

Both transformers are fine-tuned with the same recipe (Multiple Negatives Ranking Loss on title+abstract to claim pairs and citation pairs from the training split only). Retrieval quality is scored with exact FAISS search (IndexFlatIP), so the scores depend only on the embeddings; the approximate FAISS IVF index is evaluated separately as the scalable search component (its recall against exact search is reported). Metrics: Precision@K, MAP@K and NDCG@K (K = 5, 10, 20), with paired Wilcoxon signed-rank tests and Holm correction.

## Setup

In [ ]:
# Reproducible installs. The FIRST run installs the libraries and saves their exact
# versions to Drive (requirements_lock.txt). Every later run installs those same
# versions, so a new Colab session cannot silently change the results.
# torch and numpy come with Colab and are not reinstalled; their versions are checked.
import os
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/patent_thesis"
os.makedirs(WORK, exist_ok=True)
LOCK = f"{WORK}/requirements_lock.txt"
PINNED = ["sentence-transformers", "transformers", "accelerate", "datasets", "faiss-cpu",
          "nltk", "scikit-learn", "scipy"]

if os.path.exists(LOCK):
    print("Installing the locked versions from", LOCK)
    !pip -q install -r {LOCK}
else:
    !pip -q install sentence-transformers datasets accelerate faiss-cpu scikit-learn scipy matplotlib nltk
    from importlib.metadata import version
    with open(LOCK, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in PINNED) + "\n")
    with open(f"{WORK}/runtime_lock.txt", "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in ("torch", "numpy")) + "\n")
    print("Saved library versions to", LOCK)
print(open(LOCK).read())


In [ ]:
# Work inside Google Drive so nothing is lost if the Colab runtime disconnects
%cd {WORK}
# Colab updates torch/numpy over time; a different version (or GPU) can shift results slightly.
from importlib.metadata import version
if os.path.exists(f"{WORK}/runtime_lock.txt"):
    for line in open(f"{WORK}/runtime_lock.txt").read().split():
        pkg, want = line.split("==")
        have = version(pkg)
        print(f"{pkg}: locked {want}, current {have}" + ("" if have == want else "   <-- DIFFERENT: results may shift slightly"))


In [ ]:
import os, json, torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU: fine-tuning will be very slow)")
if torch.cuda.is_available() and "T4" not in torch.cuda.get_device_name(0):
    print("WARNING: the reported results were produced on a Tesla T4; a different GPU can change the numbers slightly.")
CORPUS  = "patents_G_H.jsonl"
RESULTS = "results_final"
MINILM_BASE, PATENT_BASE = "sentence-transformers/all-MiniLM-L6-v2", "AI-Growth-Lab/PatentSBERTa"
MINILM_FT,   PATENT_FT   = "models/minilm-gh", "models/patentsberta-gh"
EPOCHS, BATCH = 1, 32

def done(path):
    """True if a step already finished in an earlier session (files are in Drive)."""
    if os.path.exists(path):
        print("Already done, skipping:", path)
        return True
    return False

## Pipeline source files
These cells only write the modules to the Colab disk. Nothing runs yet.

In [ ]:
%%writefile prepare_dapfam.py
"""
prepare_dapfam.py
-----------------
Fetches the public DAPFAM benchmark (Ayaou, Cavallucci and Chibane, 2025) from
the Hugging Face Hub ENTIRELY ONLINE (no login, nothing to download by hand) and
writes the corpus file used by run_experiment.py / finetune.py.

DAPFAM (datalyes/DAPFAM_patent): family-level English patents with title,
abstract, claims, description, IPC codes; a query set; and a relation table with
citation-based relevance scores (1 = cited by / citing the query, 0 = random
negative) plus IN/OUT-domain labels.

What this script does
  1. discovers the three tables (corpus / queries / relations) on the Hub;
  2. keeps only QUERY families with at least one IPC code in Section G or H;
  3. keeps every target family linked to those queries (positives and
     negatives), so every relevance label stays resolvable;
  4. extracts the first claim as the "independent claim" (heuristic: text up to
     the start of claim 2) and writes one JSON object per patent.

Output record:
  patent_id, title, abstract, independent_claim, cpc_codes (IPC 3-char classes),
  citations (ids of positively linked families, only for queries), is_query

Usage:
    python prepare_dapfam.py --out patents_G_H.jsonl [--max-queries 0] [--sections GH]
"""

from __future__ import annotations

import argparse
import json
import random
import re
import unicodedata
from pathlib import Path

REPO = "datalyes/DAPFAM_patent"

# Column names from the DAPFAM paper (Tables 4 and 6) plus BEIR/MTEB-style aliases.
# Hyphens are normalised to underscores before matching.
Q_ID_COLS = ["query_id", "_id", "id", "qid", "family_id", "patent_id"]
T_ID_COLS = ["relevant_id", "_id", "id", "corpus_id", "doc_id", "family_id", "patent_id"]
REL_Q_COLS = ["query_id", "qid"]
REL_T_COLS = ["relevant_id", "corpus_id", "doc_id", "docid"]
TITLE_COLS = ["title_en", "title"]
ABSTRACT_COLS = ["abstract_en", "abstract", "text"]
CLAIMS_COLS = ["claims_text", "claims"]
IPC3_COLS = ["classifications_ipcr_list_first_three_chars_list", "ipc3", "ipc3_list"]
IPC_STR_COLS = ["ipcr_codes_str", "ipcr_codes", "ipc_codes"]
REL_COLS = ["relevance_score", "relevance", "score"]
KNOWN_COLS = set(Q_ID_COLS + T_ID_COLS + REL_Q_COLS + REL_T_COLS + TITLE_COLS + ABSTRACT_COLS + CLAIMS_COLS
                 + IPC3_COLS + IPC_STR_COLS + REL_COLS + ["domain_rel"])


def pick(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def clean_text(text) -> str:
    """Text normalisation (Chapter 3.2.2): Unicode NFKC, remove LaTeX-style math markup,
    control / replacement characters and collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\$\$.*?\$\$|\$[^$\n]{1,200}\$|\\\(.*?\\\)|\\\[.*?\\\]", " ", text, flags=re.S)
    text = re.sub(r"\\(begin|end)\{[^}]*\}", " ", text)
    text = re.sub(r"[\u0000-\u0008\u000b-\u001f\u007f\ufffd\u200b-\u200f\ufeff]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def first_claim(text: str, max_chars: int = 3000) -> str:
    """First claim = text before the start of claim 2 (heuristic, documented in Chapter 3)."""
    text = clean_text(text)
    if not text:
        return ""
    m = re.search(r"(?<![\d.])\b2\s*[.)]\s+(?=[A-Z])", text[3:])
    cut = (m.start() + 3) if m else len(text)
    return text[:min(cut, max_chars)].strip()


def ipc3_codes(row: dict, ipc3_col, ipc_str_col) -> list[str]:
    """IPC classes (3 chars, e.g. 'G06', 'H04') of a family."""
    raw = row.get(ipc3_col) if ipc3_col else None
    if isinstance(raw, (list, tuple)) or hasattr(raw, "tolist"):
        return sorted({str(c).strip()[:3] for c in list(raw) if str(c).strip()})
    if isinstance(raw, str) and raw.strip():
        return sorted({c[:3] for c in re.findall(r"[A-H]\d{2}", raw)})
    s = row.get(ipc_str_col) if ipc_str_col else None
    if isinstance(s, str):
        return sorted({c[:3] for c in re.findall(r"\b[A-H]\d{2}[A-Z]", s)} | {c[:3] for c in re.findall(r"\b[A-H]\d{2}\b", s)})
    return []


def build_corpus(queries, targets, relations, sections="GH", max_queries=0, seed=42):
    """Inputs are lists of dicts (queries, targets, relations).

    Returns (records, stats)."""
    q_id = pick(queries[0].keys(), Q_ID_COLS) if queries else None
    t_id = pick(targets[0].keys(), T_ID_COLS) if targets else None
    r_q = pick(relations[0].keys(), REL_Q_COLS) if relations else None
    r_t = pick(relations[0].keys(), REL_T_COLS) if relations else None
    r_s = pick(relations[0].keys(), REL_COLS) if relations else None
    if not (q_id and t_id and r_q and r_t and r_s):
        raise ValueError(f"Cannot find id/relevance columns. queries={list(queries[0]) if queries else None}, "
                         f"targets={list(targets[0]) if targets else None}, relations={list(relations[0]) if relations else None}")

    def fields(row, cols_title, cols_abs, cols_clm):
        title = row.get(pick(row.keys(), cols_title) or "", "") or ""
        abstract = row.get(pick(row.keys(), cols_abs) or "", "") or ""
        claims = row.get(pick(row.keys(), cols_clm) or "", "") or ""
        return clean_text(title), clean_text(abstract), first_claim(claims)

    def ipc_of(row):
        return ipc3_codes(row, pick(row.keys(), IPC3_COLS), pick(row.keys(), IPC_STR_COLS))

    kept_q = {}
    for row in queries:
        codes = ipc_of(row)
        if any(c[:1] in sections for c in codes):
            kept_q[str(row[q_id])] = (row, codes)
    ids = sorted(kept_q)
    if max_queries and len(ids) > max_queries:
        random.Random(seed).shuffle(ids)
        ids = sorted(ids[:max_queries])
    kept_q = {i: kept_q[i] for i in ids}

    positives: dict[str, list[str]] = {i: [] for i in kept_q}
    linked: set[str] = set()
    for rel in relations:
        qid, tid = str(rel[r_q]), str(rel[r_t])
        if qid in kept_q:
            linked.add(tid)
            try:
                positive = float(rel[r_s]) > 0
            except (TypeError, ValueError):
                positive = False
            if positive and tid != qid:
                positives[qid].append(tid)

    target_rows = {str(r[t_id]): r for r in targets}
    records: dict[str, dict] = {}
    for tid in sorted(linked):
        row = target_rows.get(tid)
        if row is None:
            continue
        title, abstract, claim = fields(row, TITLE_COLS, ABSTRACT_COLS, CLAIMS_COLS)
        records[tid] = {"patent_id": tid, "title": title, "abstract": abstract, "independent_claim": claim,
                        "cpc_codes": ipc_of(row), "citations": [], "is_query": False}
    for qid, (row, codes) in kept_q.items():
        title, abstract, claim = fields(row, TITLE_COLS, ABSTRACT_COLS, CLAIMS_COLS)
        rec = records.get(qid, {})
        rec.update({"patent_id": qid, "title": title, "abstract": abstract, "independent_claim": claim,
                    "cpc_codes": codes or rec.get("cpc_codes", []), "is_query": True})
        records[qid] = rec
    for qid, pos in positives.items():
        records[qid]["citations"] = sorted({p for p in pos if p in records})

    out = [r for r in records.values() if r["title"] or r["abstract"]]
    ids_out = {r["patent_id"] for r in out}
    for r in out:
        r["citations"] = [c for c in r["citations"] if c in ids_out]
    nq = sum(r["is_query"] for r in out)
    stats = {"patents": len(out), "queries": nq,
             "queries_with_positive": sum(1 for r in out if r["is_query"] and r["citations"]),
             "positive_links": sum(len(r["citations"]) for r in out)}
    return out, stats


def _rows(dsdict, keep_all=False):
    """All splits of a dataset concatenated, hyphens in column names turned into underscores.
    The huge description column is never loaded unless keep_all is set."""
    out = []
    for split in dsdict.keys():
        ds = dsdict[split]
        wanted = [c for c in ds.column_names if keep_all or c.replace("-", "_") in KNOWN_COLS]
        for row in ds.select_columns(wanted).to_list():
            out.append({k.replace("-", "_"): v for k, v in row.items()})
    return out


def load_tables(repo=REPO, inspect=False):
    """Download (online, cached by Hugging Face) and return queries, targets, relations as lists of dicts."""
    from datasets import get_dataset_config_names, load_dataset

    configs = get_dataset_config_names(repo)
    print("Configs on the Hub:", configs)

    def find(*keys, exclude=()):
        for c in configs:
            if c not in exclude and any(k in c.lower() for k in keys):
                return c
        return None

    cfg = {"queries": find("quer"), "targets": find("corpus", "target", "document")}
    cfg["relations"] = find("relation", "eval", "qrel", "relev") or find("default") \
        or next((c for c in configs if c not in cfg.values()), None)
    if not all(cfg.values()):
        raise SystemExit(f"Could not map the dataset configs {configs} to queries/targets/relations: {cfg}. "
                         f"Open https://huggingface.co/datasets/{repo} and edit the keywords in load_tables().")
    print("Config mapping:", cfg)
    tables = {}
    for name, c in cfg.items():
        dsdict = load_dataset(repo, c)
        rows = _rows(dsdict)
        print(f"  {name}: config={c!r}, splits={list(dsdict.keys())}, rows={len(rows)}")
        if rows:
            print("     columns used:", list(rows[0].keys()))
            if inspect:
                print("     first row :", {k: (str(v)[:70] + "...") if len(str(v)) > 70 else v for k, v in rows[0].items()})
        tables[name] = rows
    return tables["queries"], tables["targets"], tables["relations"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", type=Path, default=Path("patents_G_H.jsonl"))
    ap.add_argument("--sections", default="GH", help="IPC sections to keep for queries (default GH)")
    ap.add_argument("--max-queries", type=int, default=0, help="0 = all matching queries")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--inspect", action="store_true", help="only print configs, columns and one sample row")
    args = ap.parse_args()

    queries, targets, relations = load_tables(inspect=args.inspect)
    if args.inspect:
        return
    records, stats = build_corpus(queries, targets, relations, args.sections, args.max_queries, args.seed)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    with args.out.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with_claim = sum(1 for r in records if r["independent_claim"])
    print("Wrote", args.out, stats, f"| with a claim: {with_claim}/{len(records)}")
    if with_claim == 0:
        print("[warn] no claims found: fine-tuning will use citation pairs only (check CLAIMS_COLS).")
    if stats["queries_with_positive"] == 0:
        raise SystemExit("No query has a positive relevance link: check the column mapping above.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evaluation.py
"""
evaluation.py
-------------
Information Retrieval metrics from Proposal Section 3.5:
Precision@K, Mean Average Precision (MAP), NDCG@K, plus paired
significance testing between two systems.

Input conventions
-----------------
run   : dict[query_id -> list[doc_id]]   ranked best-first
qrels : dict[query_id -> set[doc_id]]    relevant docs (binary relevance)
"""

from __future__ import annotations

import math
from typing import Mapping, Sequence

import numpy as np


def precision_at_k(ranked: Sequence[str], relevant: set[str], k: int) -> float:
    if k <= 0:
        raise ValueError("k must be positive")
    top = ranked[:k]
    return sum(1 for d in top if d in relevant) / k


def average_precision(ranked: Sequence[str], relevant: set[str],
                      k: int | None = None) -> float:
    """AP@k (binary relevance): sum of Precision@i at each relevant rank i <= k,
    divided by |R_q| (total relevant documents for the query)."""
    if not relevant:
        return 0.0
    if k is not None:
        ranked = ranked[:k]
    hits, score = 0, 0.0
    for i, d in enumerate(ranked, start=1):
        if d in relevant:
            hits += 1
            score += hits / i
    return score / len(relevant)


def ndcg_at_k(ranked: Sequence[str], relevant: set[str], k: int) -> float:
    dcg = sum(
        1.0 / math.log2(i + 1)
        for i, d in enumerate(ranked[:k], start=1)
        if d in relevant
    )
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def per_query_scores(
    run: Mapping[str, Sequence[str]],
    qrels: Mapping[str, set[str]],
    k: int = 10,
) -> dict[str, dict[str, float]]:
    """Per-query metric values (needed for paired significance tests).
    Queries without any relevant document are skipped."""
    out: dict[str, dict[str, float]] = {}
    for qid, relevant in qrels.items():
        if not relevant or qid not in run:
            continue
        ranked = run[qid]
        out[qid] = {
            f"P@{k}": precision_at_k(ranked, relevant, k),
            "AP": average_precision(ranked, relevant, k),
            f"NDCG@{k}": ndcg_at_k(ranked, relevant, k),
        }
    return out


def summarise(per_query: Mapping[str, Mapping[str, float]]) -> dict[str, float]:
    """Mean of each metric across queries (MAP = mean of AP)."""
    if not per_query:
        return {}
    metrics = next(iter(per_query.values())).keys()
    summary = {m: float(np.mean([q[m] for q in per_query.values()])) for m in metrics}
    summary["MAP"] = summary.pop("AP")
    summary["n_queries"] = len(per_query)
    return summary


def paired_significance(
    scores_a: Mapping[str, Mapping[str, float]],
    scores_b: Mapping[str, Mapping[str, float]],
    metric: str,
) -> dict[str, float]:
    """Paired Wilcoxon signed-rank test on per-query scores of system A vs B.
    Returns mean difference (A - B), statistic and p-value."""
    from scipy.stats import wilcoxon

    common = sorted(set(scores_a) & set(scores_b))
    a = np.array([scores_a[q][metric] for q in common])
    b = np.array([scores_b[q][metric] for q in common])
    diff = a - b
    result = {"n_pairs": len(common), "mean_diff": float(diff.mean())}
    if len(common) < 6 or np.allclose(diff, 0):
        result.update(statistic=float("nan"), p_value=float("nan"))
        return result
    stat, p = wilcoxon(a, b)
    result.update(statistic=float(stat), p_value=float(p))
    return result

In [ ]:
%%writefile finetune.py
"""
finetune.py
-----------
Domain adaptation (Chapter 3). Fine-tunes a sentence-transformers model
(all-MiniLM-L6-v2 or PatentSBERTa) on (anchor, positive) pairs built from the
TRAINING split of the CPC/IPC G and H corpus, with in-batch negatives:

    anchor   = title + abstract
    positive = first (independent) claim of the same patent, and, with
               --use-citations, the title + abstract of a linked training patent
    negative = the positives of the other pairs in the batch (implicit)

Loss: MultipleNegativesRankingLoss (scale 20, i.e. temperature 0.05).
Optimiser: AdamW, lr 2e-5, linear warm-up over the first 10% of steps.

Usage:
    python finetune.py --corpus patents_G_H.jsonl --train-ids results/train_ids.txt \
        --base-model sentence-transformers/all-MiniLM-L6-v2 --out models/minilm-gh --use-citations
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import random
from pathlib import Path

# Must be set before torch initialises CUDA: makes cuBLAS matrix products deterministic.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

logger = logging.getLogger("finetune")


def set_determinism(seed: int) -> None:
    """Seed every random number generator and force deterministic GPU kernels,
    so that re-running fine-tuning on the same GPU and library versions gives the
    same model. (warn_only: the few ops without a deterministic version just warn.)"""
    import numpy as np
    import torch
    from transformers import set_seed

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def environment_info() -> dict:
    """Library versions and GPU, saved next to the model to document the run."""
    from importlib.metadata import version
    import torch

    info = {p: version(p) for p in ("torch", "transformers", "sentence-transformers", "numpy")}
    info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    return info


def load_pairs(corpus_path: Path, allowed_ids: set[str],
               use_citations: bool = False) -> list[tuple[str, str]]:
    """(anchor, positive) pairs from TRAINING patents only. A citation pair is
    used only when BOTH patents are in the training split, so no test query or
    test relevance link ever reaches fine-tuning."""
    records = []
    with corpus_path.open(encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["patent_id"] in allowed_ids:
                records.append(rec)
    summary = {r["patent_id"]: f"{r['title']} {r['abstract']}".strip() for r in records}
    pairs = []
    for rec in records:
        anchor = summary[rec["patent_id"]]
        claim = (rec.get("independent_claim") or "").strip()
        if anchor and claim:
            pairs.append((anchor, claim))
        if use_citations:
            for cited in rec.get("citations", []):
                if cited in summary and cited != rec["patent_id"] and summary[cited]:
                    pairs.append((anchor, summary[cited]))
    return pairs


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", type=Path, required=True)
    ap.add_argument("--train-ids", type=Path, required=True,
                    help="text file, one patent_id per line (training split only)")
    ap.add_argument("--base-model", required=True)
    ap.add_argument("--use-citations", action="store_true")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--max-seq-length", type=int, default=256)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

    import torch
    from sentence_transformers import InputExample, SentenceTransformer, losses
    from torch.utils.data import DataLoader

    set_determinism(args.seed)

    train_ids = set(args.train_ids.read_text().split())
    pairs = load_pairs(args.corpus, train_ids, args.use_citations)
    logger.info("Base model: %s | training pairs: %d", args.base_model, len(pairs))
    if len(pairs) < args.batch_size:
        raise SystemExit("Not enough training pairs for the chosen batch size.")

    model = SentenceTransformer(args.base_model)
    model.max_seq_length = args.max_seq_length
    examples = [InputExample(texts=[a, p]) for a, p in pairs]
    loader = DataLoader(examples, shuffle=True, batch_size=args.batch_size, drop_last=True,
                        generator=torch.Generator().manual_seed(args.seed))
    loss = losses.MultipleNegativesRankingLoss(model, scale=20.0)

    model.fit(
        train_objectives=[(loader, loss)],
        epochs=args.epochs,
        warmup_steps=int(0.1 * len(loader) * args.epochs),
        optimizer_params={"lr": args.lr},
        use_amp=torch.cuda.is_available(),
        show_progress_bar=True,
    )
    args.out.mkdir(parents=True, exist_ok=True)
    model.save(str(args.out))
    (args.out / "training_info.json").write_text(json.dumps(
        {"base_model": args.base_model, "pairs": len(pairs), "epochs": args.epochs,
         "batch_size": args.batch_size, "lr": args.lr, "max_seq_length": args.max_seq_length,
         "use_citations": args.use_citations, "seed": args.seed,
         "environment": environment_info()}, indent=2))
    logger.info("Saved fine-tuned model to %s", args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile run_experiment.py
"""
run_experiment.py
-----------------
Retrieval experiment (Chapter 3):

  BM25   : Okapi BM25, k1=1.2, b=0.75, lowercase + Porter stemming
  Dense  : sentence-transformers models (base and fine-tuned). Retrieval quality is
           scored with EXACT search (FAISS IndexFlatIP, inner product on L2-normalised
           vectors == cosine similarity), so the scores depend only on the embeddings.
  IVF    : --ivf-check also builds the approximate FAISS IndexIVFFlat index
           (nlist=256, nprobe=16) and reports its recall of the exact top-K and its
           cost (ivf_recall.json); this is the scalable search component.
  Metrics: Precision@K, MAP@K, NDCG@K for K in {5, 10, 20}
  Timing : wall-clock time of BM25 indexing/search and of encoding + FAISS per model (timings.json)
  Tests  : paired Wilcoxon signed-rank tests on per-query scores
           (every dense model vs BM25, and every fine-tuned model vs its
           base model), with Holm correction for multiple comparisons

Models are given as "Label=path_or_hub_id". Comparisons between two models
are given as "Label A|Label B" (A is compared with B).

Step 1, create the patent-level split only:
    python run_experiment.py --corpus patents_G_H.jsonl --split-only --out results

Step 2, evaluate everything:
    python run_experiment.py --corpus patents_G_H.jsonl --out results \
        --models "MiniLM (base)=sentence-transformers/all-MiniLM-L6-v2" \
                 "MiniLM (fine-tuned)=models/minilm-gh" \
                 "PatentSBERTa (base)=AI-Growth-Lab/PatentSBERTa" \
                 "PatentSBERTa (fine-tuned)=models/patentsberta-gh" \
        --pairs "MiniLM (fine-tuned)|MiniLM (base)" \
                "PatentSBERTa (fine-tuned)|PatentSBERTa (base)"
"""

from __future__ import annotations

import argparse
import csv
import json
import logging
import os
import random
import re
import time
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import numpy as np

from evaluation import paired_significance, per_query_scores, summarise

# Deterministic cuBLAS kernels (must be set before torch initialises CUDA).
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
SEED = 42

logger = logging.getLogger("experiment")
BM25_NAME = "BM25"
TIMINGS: dict[str, dict] = {}


@dataclass(frozen=True)
class Patent:
    patent_id: str
    title: str
    abstract: str

    @property
    def text(self) -> str:
        return f"{self.title} {self.abstract}".strip()


# ------------------------------------------------------------------ data
def load_corpus(path: Path) -> tuple[list[Patent], dict[str, list[str]], set[str]]:
    """Returns patents, relevance links per patent, and the ids flagged as queries."""
    patents, citations, query_ids = [], {}, set()
    with path.open(encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            patents.append(Patent(r["patent_id"], r["title"], r["abstract"]))
            citations[r["patent_id"]] = r.get("citations", [])
            if r.get("is_query"):
                query_ids.add(r["patent_id"])
    return patents, citations, query_ids


def split_by_queries(ids: list[str], citations: dict[str, list[str]], query_ids: set[str],
                     test_fraction: float, seed: int) -> tuple[set[str], set[str]]:
    """Patent-level split of the query families. A test query and every family
    linked to it are removed from training, so no relevance link leaks into
    fine-tuning."""
    rng = random.Random(seed)
    qs = sorted(query_ids)
    rng.shuffle(qs)
    test = set(qs[:max(1, int(len(qs) * test_fraction))])
    held_out = set(test)
    for q in test:
        held_out.update(citations.get(q, []))
    return set(ids) - held_out, test


def build_qrels(test_ids: set[str], citations: dict[str, list[str]],
                corpus_ids: set[str]) -> dict[str, set[str]]:
    qrels = {}
    for pid in test_ids:
        rel = {c for c in citations.get(pid, []) if c in corpus_ids and c != pid}
        if rel:
            qrels[pid] = rel
    return qrels


# ------------------------------------------------------------ BM25
_TOKEN = re.compile(r"[a-z0-9]+")
_stemmer = None


@lru_cache(maxsize=None)
def _stem(word: str) -> str:
    global _stemmer
    if _stemmer is None:
        from nltk.stem import PorterStemmer
        _stemmer = PorterStemmer()
    return _stemmer.stem(word)


def tokenize(text: str) -> list[str]:
    return [_stem(t) for t in _TOKEN.findall(text.lower())]


class Bm25Index:
    """Okapi BM25 via sparse matrices:
    score(q,d) = sum_t idf(t) * tf*(k1+1) / (tf + k1*(1-b+b*|d|/avgdl))."""

    def __init__(self, texts: list[str], k1: float = 1.2, b: float = 0.75) -> None:
        from scipy import sparse
        from sklearn.feature_extraction.text import CountVectorizer

        self._cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
        tf = self._cv.fit_transform(texts).tocoo().astype(np.float32)
        n = tf.shape[0]
        dl = np.asarray(tf.tocsr().sum(axis=1)).ravel()
        avgdl = float(dl.mean())
        df = np.asarray((tf.tocsr() > 0).sum(axis=0)).ravel()
        idf = np.log((n - df + 0.5) / (df + 0.5) + 1.0)
        denom = tf.data + k1 * (1.0 - b + b * dl[tf.row] / avgdl)
        data = tf.data * (k1 + 1.0) / denom * idf[tf.col]
        self._w = sparse.csr_matrix((data, (tf.row, tf.col)), shape=tf.shape).T.tocsr()

    def scores(self, query_texts: list[str]) -> np.ndarray:
        q = self._cv.transform(query_texts).astype(np.float32)
        return (q @ self._w).toarray()


def top_ranked(row: np.ndarray, top_k: int) -> np.ndarray:
    top_k = min(top_k, len(row) - 1)
    part = np.argpartition(-row, top_k)[:top_k]
    return part[np.argsort(-row[part])]


def bm25_run(texts: list[str], ids: list[str], query_idx: list[int],
             top_k: int, batch: int = 256) -> dict[str, list[str]]:
    t0 = time.perf_counter()
    index = Bm25Index(texts)
    TIMINGS["BM25"] = {"index_build_s": round(time.perf_counter() - t0, 2)}
    t1 = time.perf_counter()
    run: dict[str, list[str]] = {}
    for s in range(0, len(query_idx), batch):
        chunk = query_idx[s:s + batch]
        sc = index.scores([texts[i] for i in chunk])
        for row, qi in zip(sc, chunk):
            row[qi] = -np.inf  # a patent is never its own answer
            run[ids[qi]] = [ids[j] for j in top_ranked(row, top_k)]
    TIMINGS["BM25"]["search_s"] = round(time.perf_counter() - t1, 2)
    TIMINGS["BM25"]["n_queries"] = len(query_idx)
    return run


# ------------------------------------------------------------ dense + FAISS
def encode(model_path: str, texts: list[str], batch_size: int = 64, label: str = "") -> np.ndarray:
    import torch
    from sentence_transformers import SentenceTransformer

    torch.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    logger.info("Encoding with %s", model_path)
    model = SentenceTransformer(model_path)
    model.eval()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    emb = model.encode(texts, batch_size=batch_size, show_progress_bar=True,
                       convert_to_numpy=True).astype("float32")
    secs = time.perf_counter() - t0
    TIMINGS[label or model_path] = {
        "parameters": int(sum(p.numel() for p in model.parameters())),
        "dimension": int(emb.shape[1]), "max_seq_length": int(model.max_seq_length),
        "n_docs": len(texts), "encode_s": round(secs, 2), "docs_per_s": round(len(texts) / secs, 1),
        "ms_per_doc": round(1000 * secs / len(texts), 2),
        "device": "cuda" if torch.cuda.is_available() else "cpu"}
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return emb


def _unit(emb: np.ndarray) -> np.ndarray:
    import faiss

    x = np.ascontiguousarray(emb, dtype="float32").copy()
    faiss.normalize_L2(x)  # inner product on unit vectors == cosine similarity
    return x


def _build_index(x: np.ndarray, kind: str, nlist: int = 256, nprobe: int = 16):
    import faiss

    d = x.shape[1]
    if kind == "exact":
        index = faiss.IndexFlatIP(d)
    else:
        nlist = max(1, min(nlist, len(x) // 39))  # FAISS needs >=39 points per centroid
        quantizer = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
        index.cp.seed = SEED  # fixed k-means initialisation
        index.train(x)
        index.nprobe = min(nprobe, nlist)
    index.add(x)
    return index


def _search(index, x: np.ndarray, query_idx: list[int], top_k: int) -> list[list[int]]:
    """Top-k row indices per query, the query itself removed."""
    _, nbrs = index.search(x[query_idx], top_k + 1)
    return [[int(j) for j in row if j != -1 and j != qi][:top_k] for qi, row in zip(query_idx, nbrs)]


def faiss_run(emb: np.ndarray, ids: list[str], query_idx: list[int], top_k: int,
              kind: str = "exact", label: str = "") -> dict[str, list[str]]:
    t0 = time.perf_counter()
    x = _unit(emb)
    index = _build_index(x, kind)
    logger.info("FAISS %s index: %d vectors, dim=%d", kind, len(x), x.shape[1])
    hits = _search(index, x, query_idx, top_k)
    if label:
        TIMINGS.setdefault(label, {})["faiss_build_search_s"] = round(time.perf_counter() - t0, 2)
    return {ids[qi]: [ids[j] for j in row] for qi, row in zip(query_idx, hits)}


def ivf_recall(emb: np.ndarray, query_idx: list[int], ks: list[int],
               nlist: int = 256, nprobe: int = 16) -> dict:
    """How much of the exact top-K the approximate IVF index finds (recall@K, averaged over
    queries), plus build and search times of both indexes."""
    x = _unit(emb)
    depth = max(ks)
    t0 = time.perf_counter()
    exact = _build_index(x, "exact")
    t1 = time.perf_counter()
    exact_hits = _search(exact, x, query_idx, depth)
    t2 = time.perf_counter()
    ivf = _build_index(x, "ivf", nlist, nprobe)
    t3 = time.perf_counter()
    ivf_hits = _search(ivf, x, query_idx, depth)
    t4 = time.perf_counter()
    out = {"nlist": ivf.nlist, "nprobe": ivf.nprobe, "n_queries": len(query_idx),
           "exact_build_s": round(t1 - t0, 3), "exact_search_s": round(t2 - t1, 3),
           "ivf_build_s": round(t3 - t2, 3), "ivf_search_s": round(t4 - t3, 3)}
    for k in ks:
        out[f"recall@{k}"] = round(float(np.mean(
            [len(set(e[:k]) & set(a[:k])) / k for e, a in zip(exact_hits, ivf_hits)])), 4)
    return out


# ------------------------------------------------------------ statistics
def holm(p_values: list[float]) -> list[float]:
    """Holm-Bonferroni adjusted p-values (NaN entries are left as NaN)."""
    idx = [i for i, p in enumerate(p_values) if p == p]
    order = sorted(idx, key=lambda i: p_values[i])
    m, running, adj = len(order), 0.0, [float("nan")] * len(p_values)
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p_values[i]))
        adj[i] = running
    return adj


def parse_models(specs: list[str]) -> dict[str, str]:
    models = {}
    for s in specs:
        label, sep, path = s.partition("=")
        if not sep:
            label, path = s, s
        models[label.strip()] = path.strip()
    return models


# ------------------------------------------------------------------ main
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", type=Path, required=True)
    ap.add_argument("--models", nargs="*", default=[], help='"Label=path_or_hub_id"')
    ap.add_argument("--pairs", nargs="*", default=[], help='"Label A|Label B" extra comparisons')
    ap.add_argument("--split-only", action="store_true", help="write train/test ids and stop")
    ap.add_argument("--test-fraction", type=float, default=0.3)
    ap.add_argument("--ks", type=int, nargs="+", default=[5, 10, 20])
    ap.add_argument("--index", choices=["ivf", "exact"], default="exact",
                    help="index used to SCORE retrieval quality (default exact)")
    ap.add_argument("--ivf-check", action="store_true",
                    help="also measure IVF recall of the exact top-K and its cost (ivf_recall.json)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", type=Path, default=Path("results"))
    args = ap.parse_args()

    global SEED
    SEED = args.seed
    random.seed(args.seed)
    np.random.seed(args.seed)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
    args.out.mkdir(parents=True, exist_ok=True)

    patents, citations, query_ids = load_corpus(args.corpus)
    if not query_ids:
        raise SystemExit("The corpus has no records flagged is_query: rebuild it with prepare_dapfam.py.")
    ids = [p.patent_id for p in patents]
    train_ids, test_ids = split_by_queries(ids, citations, query_ids, args.test_fraction, args.seed)
    (args.out / "train_ids.txt").write_text("\n".join(sorted(train_ids)))
    (args.out / "test_ids.txt").write_text("\n".join(sorted(test_ids)))
    qrels = build_qrels(test_ids, citations, set(ids))
    query_idx = [i for i, pid in enumerate(ids) if pid in qrels]
    split_stats = {"corpus": len(patents), "train_patents": len(train_ids),
                   "test_queries": len(test_ids), "test_queries_with_relevant": len(qrels),
                   "relevant_links_in_test": sum(len(v) for v in qrels.values())}
    (args.out / "split_stats.json").write_text(json.dumps(split_stats, indent=2))
    logger.info("Split: %s", split_stats)
    if not qrels:
        raise SystemExit("No test query has an in-corpus relevant patent.")
    if args.split_only:
        return

    texts = [p.text for p in patents]
    depth = max(args.ks)
    runs = {BM25_NAME: bm25_run(texts, ids, query_idx, depth)}
    recall = {}
    for label, path in parse_models(args.models).items():
        emb = encode(path, texts, label=label)
        runs[label] = faiss_run(emb, ids, query_idx, depth, kind=args.index, label=label)
        if args.ivf_check:
            recall[label] = ivf_recall(emb, query_idx, args.ks)
            logger.info("IVF vs exact, %s: %s", label, recall[label])
    (args.out / "timings.json").write_text(json.dumps(TIMINGS, indent=2))
    if recall:
        (args.out / "ivf_recall.json").write_text(json.dumps(recall, indent=2))

    comparisons = [(n, BM25_NAME) for n in runs if n != BM25_NAME]
    for spec in args.pairs:
        a, _, b = (s.strip() for s in spec.partition("|"))
        if a in runs and b in runs:
            comparisons.append((a, b))
        else:
            logger.warning("Skipping comparison %r: unknown label", spec)

    rows, sig_rows, pq_rows = [], [], []
    for k in args.ks:
        per_system = {n: per_query_scores(r, qrels, k) for n, r in runs.items()}
        for n, pq in per_system.items():
            for qid, sc in pq.items():
                pq_rows.append({"query_id": qid, "n_relevant": len(qrels[qid]), "system": n, "K": k,
                                "P@K": round(sc[f"P@{k}"], 6), "AP@K": round(sc["AP"], 6),
                                "NDCG@K": round(sc[f"NDCG@{k}"], 6)})
        for n, pq in per_system.items():
            s = summarise(pq)
            rows.append({"system": n, "K": k, "P@K": round(s[f"P@{k}"], 4),
                         "MAP@K": round(s["MAP"], 4), "NDCG@K": round(s[f"NDCG@{k}"], 4),
                         "n_queries": s["n_queries"]})
            logger.info("%-28s K=%-2d P@K=%.4f MAP@K=%.4f NDCG@K=%.4f", n, k,
                        s[f"P@{k}"], s["MAP"], s[f"NDCG@{k}"])
        for a, b in comparisons:
            for metric, label in ((f"P@{k}", "P@K"), ("AP", "MAP@K"), (f"NDCG@{k}", "NDCG@K")):
                r = paired_significance(per_system[a], per_system[b], metric)
                sig_rows.append({"system": a, "compared_with": b, "K": k, "metric": label, **r})

    for r, p in zip(sig_rows, holm([r["p_value"] for r in sig_rows])):
        r["p_holm"] = p
        r["significant_0.05"] = bool(p == p and p < 0.05)

    with (args.out / "results.csv").open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    with (args.out / "significance.csv").open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(sig_rows[0]))
        w.writeheader()
        w.writerows(sig_rows)
    with (args.out / "per_query_scores.csv").open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(pq_rows[0]))
        w.writeheader()
        w.writerows(pq_rows)
    logger.info("Wrote results.csv, significance.csv and per_query_scores.csv to %s", args.out)


if __name__ == "__main__":
    main()

## Step 1: Load the data (DAPFAM, Hugging Face Hub)
`--inspect` prints the dataset structure first. Then the corpus is built: query families with an IPC code in section G or H, plus every target family linked to them.

In [ ]:
if not done(CORPUS):
    !python prepare_dapfam.py --inspect

In [ ]:
if not done(CORPUS):
    !python prepare_dapfam.py --out {CORPUS}

In [ ]:
recs = [json.loads(l) for l in open(CORPUS)]
queries = [r for r in recs if r["is_query"]]
print("patents in corpus:", len(recs))
print("query families (IPC section G/H):", len(queries))
print("queries with at least one positive link:", sum(1 for r in queries if r["citations"]))
print("positive links in total:", sum(len(r["citations"]) for r in queries))
print("records with an extracted claim:", sum(1 for r in recs if r["independent_claim"]))

## Step 2: Patent-level train/test split
30% of the query families become test queries. They and every patent linked to them are excluded from fine-tuning.

In [ ]:
if not done(f"{RESULTS}/train_ids.txt"):
    !python run_experiment.py --corpus {CORPUS} --split-only --test-fraction 0.3 --out {RESULTS}
print(open(f"{RESULTS}/split_stats.json").read())

## Step 3: Fine-tune all-MiniLM-L6-v2 (training split only)
Fine-tuning runs only if `models/minilm-gh` is missing on Drive. **Do not delete the `models/` folder**: retraining is the step that changes the results. With the fixed seed and deterministic settings, a retrain on the same GPU and locked library versions should give the same model.

In [ ]:
if not done(f"{MINILM_FT}/training_info.json"):
    !python finetune.py --corpus {CORPUS} --train-ids {RESULTS}/train_ids.txt --base-model {MINILM_BASE} --out {MINILM_FT} --epochs {EPOCHS} --batch-size {BATCH} --use-citations

## Step 4: Fine-tune PatentSBERTa (training split only, same recipe)

In [ ]:
if not done(f"{PATENT_FT}/training_info.json"):
    !python finetune.py --corpus {CORPUS} --train-ids {RESULTS}/train_ids.txt --base-model {PATENT_BASE} --out {PATENT_FT} --epochs {EPOCHS} --batch-size {BATCH} --use-citations

## Step 5: Evaluate all five systems on the test queries
Retrieval quality is scored with **exact** search, so the results depend only on the embeddings (no clustering, no seed). `--ivf-check` additionally builds the approximate IVF index (256 cells, 16 searched) and measures how much of the exact top-K it finds.

In [ ]:
!python run_experiment.py --corpus {CORPUS} --test-fraction 0.3 --out {RESULTS} --index exact --ivf-check \
   --models "MiniLM (base)={MINILM_BASE}" \
            "MiniLM (fine-tuned)={MINILM_FT}" \
            "PatentSBERTa (base)={PATENT_BASE}" \
            "PatentSBERTa (fine-tuned)={PATENT_FT}" \
   --pairs "MiniLM (fine-tuned)|MiniLM (base)" \
           "PatentSBERTa (fine-tuned)|PatentSBERTa (base)" \
           "PatentSBERTa (base)|MiniLM (base)" \
           "PatentSBERTa (fine-tuned)|MiniLM (fine-tuned)"

## Results
Everything below is read from the files written in Step 5.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
ORDER = ["BM25", "MiniLM (base)", "MiniLM (fine-tuned)", "PatentSBERTa (base)", "PatentSBERTa (fine-tuned)"]
res = pd.read_csv(f"{RESULTS}/results.csv")
res["system"] = pd.Categorical(res["system"], ORDER, ordered=True)
table = res.pivot_table(index="system", columns="K", values=["P@K", "MAP@K", "NDCG@K"], observed=False)
display(table.round(4))
table.round(4).to_csv(f"{RESULTS}/results_table.csv")

In [ ]:
sig = pd.read_csv(f"{RESULTS}/significance.csv")
pd.set_option("display.max_rows", 200)
display(sig.round(4))

In [ ]:
# Fine-tuning gain: change from each base model to its fine-tuned version
gain = res.pivot_table(index="K", columns="system", values=["P@K", "MAP@K", "NDCG@K"], observed=False)
rows = []
for m in ["P@K", "MAP@K", "NDCG@K"]:
    for base, ft in [("MiniLM (base)", "MiniLM (fine-tuned)"), ("PatentSBERTa (base)", "PatentSBERTa (fine-tuned)")]:
        for k in gain.index:
            b, f = gain.loc[k, (m, base)], gain.loc[k, (m, ft)]
            rows.append({"model": base.split(" ")[0], "metric": m, "K": k, "base": b, "fine-tuned": f,
                         "abs_gain": f - b, "rel_gain_%": 100 * (f - b) / b if b else float("nan")})
gain_df = pd.DataFrame(rows)
display(gain_df.round(4))
gain_df.round(4).to_csv(f"{RESULTS}/finetuning_gain.csv", index=False)

### Report figures (Figures 6.1, 6.2 and 6.3)
The same code produced the figures in the report.

In [ ]:
SYSTEMS = [("BM25", "BM25", "#8c8c8c"),
           ("MiniLM (base)", "MiniLM (base)", "#9ecae1"),
           ("MiniLM (fine-tuned)", "MiniLM (fine-tuned)", "#2b7bba"),
           ("PatentSBERTa (base)", "PatentSBERTa (base)", "#f5b971"),
           ("PatentSBERTa (fine-tuned)", "PatentSBERTa (fine-tuned)", "#e0862a")]
KS = [5, 10, 20]
val = {(r.system, int(r.K)): r for r in res.itertuples(index=False)}
col = {"P@K": "P@K", "MAP@K": "MAP@K", "NDCG@K": "NDCG@K"}
for metric, fname, ylab in (("P@K", "fig6_1_precision.png", "Precision@K"),
                            ("MAP@K", "fig6_2_map.png", "MAP@K"),
                            ("NDCG@K", "fig6_3_ndcg.png", "NDCG@K")):
    get = lambda s, k: float(res[(res.system == s) & (res.K == k)][metric].iloc[0])
    fig, ax = plt.subplots(figsize=(7.5, 4.0), dpi=200)
    w = 0.16
    for i, (key, label, c) in enumerate(SYSTEMS):
        xs = [k + (i - 2) * w for k in range(3)]
        vals = [get(key, K) for K in KS]
        bars = ax.bar(xs, vals, w, color=c, label=label)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.01, f"{v:.3f}", ha="center", va="bottom", rotation=90, fontsize=6.5)
    top = max(get(s, K) for s, _, _ in SYSTEMS for K in KS)
    ax.set_ylim(0, top * 1.38)
    ax.set_xticks(range(3)); ax.set_xticklabels([f"K={k}" for k in KS])
    ax.set_ylabel(ylab)
    ax.set_title(f"{ylab} on {int(res.n_queries.iloc[0])} held-out test queries", fontsize=10)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.legend(ncol=3, fontsize=6.8, frameon=False, loc="upper right")
    fig.tight_layout(); fig.savefig(f"{RESULTS}/{fname}"); plt.show()

### Computational cost (Table 6.5), model sizes and IVF recall
Wall-clock times measured by `run_experiment.py` in Step 5 (`timings.json`), and the recall of the approximate IVF index against exact search (`ivf_recall.json`).

In [ ]:
tim = json.load(open(f"{RESULTS}/timings.json"))
print("BM25: index build %.2f s, search of %d queries %.2f s" % (tim["BM25"]["index_build_s"], tim["BM25"]["n_queries"], tim["BM25"]["search_s"]))
cost = pd.DataFrame({k: v for k, v in tim.items() if k != "BM25"}).T
display(cost[["parameters", "dimension", "max_seq_length", "device", "encode_s", "docs_per_s", "ms_per_doc", "faiss_build_search_s"]])
print("faiss_build_search_s = exact index (IndexFlatIP) build + search of all test queries")

# Approximate IVF index vs exact search: recall of the exact top-K and cost
rec = pd.DataFrame(json.load(open(f"{RESULTS}/ivf_recall.json"))).T
display(rec)
rec.to_csv(f"{RESULTS}/ivf_recall_table.csv")

In [ ]:
import shutil
from google.colab import files
for d in (MINILM_FT, PATENT_FT):
    p = os.path.join(d, "training_info.json")
    if os.path.exists(p):
        shutil.copy(p, os.path.join(RESULTS, os.path.basename(d) + "_training_info.json"))
# The zip is saved to Drive (patent_thesis/results_final.zip), so it is safe even if the
# browser download below does not start (common during "Run all").
zip_path = shutil.make_archive(os.path.join(WORK, "results_final"), "zip", RESULTS)
print("Saved", zip_path)
try:
    files.download(zip_path)
except Exception as e:
    print("Browser download did not start; get the zip from Google Drive instead:", e)


# Supplementary experiment: longer fine-tuning (3 epochs)
**Question:** was PatentSBERTa (110M parameters) under-trained by the single-epoch recipe, and does more training close its gap to MiniLM?

Both encoders are fine-tuned again for **3 epochs** with everything else unchanged (same training split, pairs, loss, batch size, learning rate, seed). The epoch count is fixed in advance, not tuned on the test queries.

Kept separate from the main experiment on purpose:
- the new models go to **new folders** (`models/*-3ep`), so the main 1-epoch models are never touched;
- results go to **`results_3ep/`**, with their own Holm correction, so the main tables and p-values in `results_final/` do not change.

The 1-epoch models are evaluated again here as the reference; their scores should equal those in `results_final` (checked below).

Training takes about 3x as long as the main fine-tuning. Each model is saved to Drive when it finishes, so if Colab disconnects, re-run this section and finished models are skipped.

In [ ]:
EPOCHS_LONG  = 3
MINILM_FT3   = "models/minilm-gh-3ep"
PATENT_FT3   = "models/patentsberta-gh-3ep"
RESULTS_LONG = "results_3ep"

# Same training split as the main experiment
if not done(f"{MINILM_FT3}/training_info.json"):
    !python finetune.py --corpus {CORPUS} --train-ids {RESULTS}/train_ids.txt --base-model {MINILM_BASE} --out {MINILM_FT3} --epochs {EPOCHS_LONG} --batch-size {BATCH} --use-citations

In [ ]:
if not done(f"{PATENT_FT3}/training_info.json"):
    !python finetune.py --corpus {CORPUS} --train-ids {RESULTS}/train_ids.txt --base-model {PATENT_BASE} --out {PATENT_FT3} --epochs {EPOCHS_LONG} --batch-size {BATCH} --use-citations

### Evaluate 1-epoch vs 3-epoch models
Comparisons (each with P@K, MAP@K, NDCG@K at K = 5, 10, 20; Holm-corrected within this experiment only): every model vs BM25, 3 epochs vs 1 epoch for each encoder, and PatentSBERTa vs MiniLM at 3 epochs.

In [ ]:
# Exact search, as in the main experiment, so the 1-epoch rows must equal results_final
if not done(f"{RESULTS_LONG}/results.csv"):
    !python run_experiment.py --corpus {CORPUS} --test-fraction 0.3 --out {RESULTS_LONG} --index exact \
       --models "MiniLM (1 epoch)={MINILM_FT}" \
                "MiniLM (3 epochs)={MINILM_FT3}" \
                "PatentSBERTa (1 epoch)={PATENT_FT}" \
                "PatentSBERTa (3 epochs)={PATENT_FT3}" \
       --pairs "MiniLM (3 epochs)|MiniLM (1 epoch)" \
               "PatentSBERTa (3 epochs)|PatentSBERTa (1 epoch)" \
               "PatentSBERTa (3 epochs)|MiniLM (3 epochs)"

In [ ]:
ORDER3 = ["BM25", "MiniLM (1 epoch)", "MiniLM (3 epochs)", "PatentSBERTa (1 epoch)", "PatentSBERTa (3 epochs)"]
res3 = pd.read_csv(f"{RESULTS_LONG}/results.csv")
res3["system"] = pd.Categorical(res3["system"], ORDER3, ordered=True)
table3 = res3.pivot_table(index="system", columns="K", values=["P@K", "MAP@K", "NDCG@K"], observed=False)
display(table3.round(4))
table3.round(4).to_csv(f"{RESULTS_LONG}/results_table.csv")

# Sanity check: the same split and the 1-epoch models must reproduce the main results exactly
main = pd.read_csv(f"{RESULTS}/results.csv").set_index(["system", "K"])
same = {"BM25": "BM25", "MiniLM (1 epoch)": "MiniLM (fine-tuned)", "PatentSBERTa (1 epoch)": "PatentSBERTa (fine-tuned)"}
ok = all(
    abs(r[m] - main.loc[(same[r["system"]], r["K"]), m]) < 1e-9
    for _, r in res3[res3.system.isin(same.keys())].iterrows() for m in ("P@K", "MAP@K", "NDCG@K"))
print("1-epoch reference matches results_final:", ok)

In [ ]:
# Effect of training longer (1 -> 3 epochs), with Holm-adjusted p-values from this experiment
sig3 = pd.read_csv(f"{RESULTS_LONG}/significance.csv")
g = res3.set_index(["system", "K"])
rows = []
for enc in ("MiniLM", "PatentSBERTa"):
    a, b = f"{enc} (3 epochs)", f"{enc} (1 epoch)"
    for m in ("P@K", "MAP@K", "NDCG@K"):
        for k in (5, 10, 20):
            one, three = g.loc[(b, k), m], g.loc[(a, k), m]
            s = sig3[(sig3.system == a) & (sig3.compared_with == b) & (sig3.K == k) & (sig3.metric == m)].iloc[0]
            rows.append({"encoder": enc, "metric": m, "K": k, "1 epoch": one, "3 epochs": three,
                         "abs_gain": three - one, "rel_gain_%": 100 * (three - one) / one,
                         "p_raw": s.p_value, "p_holm": s.p_holm, "significant": s["significant_0.05"]})
epoch_df = pd.DataFrame(rows)
display(epoch_df.round(4))
epoch_df.round(4).to_csv(f"{RESULTS_LONG}/epoch_gain.csv", index=False)

# Does the gap between the encoders close at 3 epochs?
gap = sig3[(sig3.system == "PatentSBERTa (3 epochs)") & (sig3.compared_with == "MiniLM (3 epochs)")]
display(gap[["K", "metric", "mean_diff", "p_value", "p_holm", "significant_0.05"]].round(4))
print("mean_diff < 0 means PatentSBERTa is still below MiniLM.")

In [ ]:
# Figure S.1: NDCG@K for 1 vs 3 epochs (same style as Figures 6.1-6.3)
SYSTEMS3 = [("BM25", "#8c8c8c"), ("MiniLM (1 epoch)", "#9ecae1"), ("MiniLM (3 epochs)", "#2b7bba"),
            ("PatentSBERTa (1 epoch)", "#f5b971"), ("PatentSBERTa (3 epochs)", "#e0862a")]
get3 = lambda s, k: float(res3[(res3.system == s) & (res3.K == k)]["NDCG@K"].iloc[0])
fig, ax = plt.subplots(figsize=(7.5, 4.0), dpi=200)
w = 0.16
for i, (s, c) in enumerate(SYSTEMS3):
    xs = [k + (i - 2) * w for k in range(3)]
    vals = [get3(s, K) for K in KS]
    bars = ax.bar(xs, vals, w, color=c, label=s)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.002, f"{v:.3f}", ha="center", va="bottom", rotation=90, fontsize=6.5)
ax.set_ylim(0, max(get3(s, K) for s, _ in SYSTEMS3 for K in KS) * 1.38)
ax.set_xticks(range(3)); ax.set_xticklabels([f"K={k}" for k in KS])
ax.set_ylabel("NDCG@K")
ax.set_title(f"NDCG@K after 1 and 3 epochs of fine-tuning ({int(res3.n_queries.iloc[0])} test queries)", fontsize=10)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.legend(ncol=3, fontsize=6.8, frameon=False, loc="upper right")
fig.tight_layout(); fig.savefig(f"{RESULTS_LONG}/figS1_ndcg_epochs.png"); plt.show()

In [ ]:
# Package the supplementary results (separate from results_final.zip), saved to Drive
for d in (MINILM_FT3, PATENT_FT3):
    p = os.path.join(d, "training_info.json")
    if os.path.exists(p):
        shutil.copy(p, os.path.join(RESULTS_LONG, os.path.basename(d) + "_training_info.json"))
zip_path = shutil.make_archive(os.path.join(WORK, "results_3ep"), "zip", RESULTS_LONG)
print("Saved", zip_path)
try:
    files.download(zip_path)
except Exception as e:
    print("Browser download did not start; get the zip from Google Drive instead:", e)
